In [1]:
%load_ext autoreload
%autoreload 2

import warnings
import matplotlib.pyplot as plt
import numpy as np
import glob
import xarray as xr
import gsw
# import regionate
import cartopy.crs as ccrs
# import CM4Xutils #needed to run pip install nc-time-axis
import sys
sys.path.insert(0, '/vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/src')
from src import *
import cartopy.feature as cfeature
import warnings
# from numpy import RankWarning
import cmocean as cm
warnings.filterwarnings('ignore')
import xesmf as xe
from meridional_streamfunction import * 

In [2]:
from dask_jobqueue import SLURMCluster  # setup dask cluster 
from dask.distributed import Client

log_directory="/vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/WaterMassBudgets/logs"

cluster = SLURMCluster(
    cores=36,
    processes=1,
    memory='190GB',
    walltime='03:00:00',
    queue='compute',
    interface='ib0', 
log_directory = log_directory)
print(cluster.job_script())
cluster.scale(jobs=16)

client = Client(cluster)
client

#!/usr/bin/env bash

#SBATCH -J dask-worker
#SBATCH -e /vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/WaterMassBudgets/logs/dask-worker-%J.err
#SBATCH -o /vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/WaterMassBudgets/logs/dask-worker-%J.out
#SBATCH -p compute
#SBATCH -n 1
#SBATCH --cpus-per-task=36
#SBATCH --mem=177G
#SBATCH -t 03:00:00

/vortexfs1/home/anthony.meza/miniforge3/envs/cm4x_chapter3/bin/python -m distributed.cli.dask_worker tcp://172.16.3.94:34318 --name dummy-name --nthreads 36 --memory-limit 176.95GiB --nanny --death-timeout 60 --interface ib0



Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://172.16.3.94:34318,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [3]:
import glob

In [4]:
import xarray as xr
import glob

high_res_datadir = lambda x="" : "/vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/data/ocean_month_rho2/" + x

high_res_datafiles = sorted(glob.glob(high_res_datadir("*")))

# 1. Open datasets
ds_highres_ = xr.open_mfdataset(
    high_res_datafiles,
    chunks = {"time": 1},
    data_vars="minimal",
    coords="minimal",
    compat="override",
    parallel=True,
    engine="netcdf4"
)
ds_highres_.coords["xh"] = np.arange(len(ds_highres_.coords["xh"]))
ds_highres_.coords["yh"] = np.arange(len(ds_highres_.coords["yh"]))
ds_highres_.coords["xq"] = np.arange(len(ds_highres_.coords["xq"]))
ds_highres_.coords["yq"] = np.arange(len(ds_highres_.coords["yq"]))

high_res_coords = xr.open_mfdataset("/vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/data/model/native_resolution/CM4Xp125_native_resolution_example_2010.zarr", 
                            engine = "zarr")

# 2. Merge and drop variables
ds_highres = xr.merge([ds_highres_, high_res_coords.coords])
ds_highres = ds_highres.drop_vars(["average_DT", "average_T1", "average_T2", "time_bnds"])

for var in ds_highres.variables:
    ds_highres[var].encoding.clear()

In [5]:
# 3. DYNAMIC CHUNKING: Catch every dimension
chunk_dict = {"time": 1}
ds_highres = ds_highres.chunk(chunk_dict)

for dim in ds_highres.dims:
    if dim != "time":
        chunk_dict[dim] = 500
chunk_dict["rho2_l"] = 10

# Apply the catch-all chunks
ds_highres = ds_highres.chunk(chunk_dict)

for var in ds_highres.variables:
    ds_highres[var].encoding.clear()
    
ds_highres.to_zarr(
    "/vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/data/ocean_month_rho2.zarr", 
    compute=True, 
    mode="w",
    align_chunks = True
)